<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Getting Started with PATSTAT</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Your <strong>first hands-on PATSTAT queries</strong> on EPO&nbsp;TIP &mdash; no SQL experience needed. Change one line, run the cell, read the result.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 640px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What you will do in this notebook</strong>
            <br/>Setup &nbsp;&middot;&nbsp; Connect to PATSTAT
            <br/><br/><em>Part&nbsp;1 &mdash; Search a company's patents</em> (example: Siemens)
            <br/>1 &nbsp;&middot;&nbsp; Find patents by applicant name
            <br/>2 &nbsp;&middot;&nbsp; Filter by jurisdiction
            <br/>3 &nbsp;&middot;&nbsp; Filings per jurisdiction &amp; year
            <br/>4 &nbsp;&middot;&nbsp; Patent families
            <br/>5 &nbsp;&middot;&nbsp; Families filtered by jurisdiction
            <br/><br/><em>Part&nbsp;2 &mdash; Profile an institution</em> (example: TU&nbsp;Berlin)
            <br/>1 &nbsp;&middot;&nbsp; Find the right name (<code>psn_name</code>)
            <br/>2 &nbsp;&middot;&nbsp; Portfolio overview
            <br/>3 &nbsp;&middot;&nbsp; Timeline
            <br/>4 &nbsp;&middot;&nbsp; A family under the microscope
            <br/>5 &nbsp;&middot;&nbsp; Filing strategy
            <br/>6 &nbsp;&middot;&nbsp; Technology profile
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 640px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            The defaults (<strong>Siemens</strong>, <strong>TU&nbsp;Berlin</strong>) run out of the box. To use your own,
            change the value under each <code># --- CHANGE THIS ---</code> marker. Every query runs directly on PATSTAT
            inside EPO&nbsp;TIP.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025.
    </div>
</div>

## A few PATSTAT concepts before we start

Each query builds on the previous one. These terms come up throughout:

| Term | Meaning |
|:-----|:--------|
| `person_name` | Name as delivered by the patent office (many spelling variants!) |
| `psn_name` | PATSTAT-standardised name — the best choice for organisations & universities |
| `han_name` | OECD-harmonised name — convenient, but has known errors |
| `docdb_family_id` | One invention, regardless of how many countries it was filed in |
| `applt_seq_nr > 0` | Filter for **applicants** only (not inventors) |

**Rule of thumb:** count `docdb_family_id` to count *inventions*, and always keep `applt_seq_nr > 0` so you get applicants, not inventors.

## Setup: Connect to PATSTAT

In [ ]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute a PATSTAT SQL query and return a pandas DataFrame."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    df = pd.DataFrame(res)
    print(f"Query took {time.time() - start:.1f}s - {len(df)} rows returned")
    return df
print("Ready! Proceed to next cell.")

---
# Part 1 &mdash; Search a company's patents

Start simple: find a company's applications, then slice them by jurisdiction, year, and patent family. The default example is **Siemens** — change the `APPLICANT` value in any cell to search for another company (e.g. `'%bosch%'`, `'%samsung%'`, `'%basf%'`).

---

## Query 1: Find Patents by Applicant Name

Search for all patent applications from a specific company.

**Try it:** Change `'%siemens%'` to any company you are interested in (e.g. `'%bosch%'`, `'%samsung%'`, `'%basf%'`).

In [ ]:
# --- CHANGE THIS ---
APPLICANT = '%siemens%'
# -------------------

df_q1 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    a.appln_nr_epodoc AS application_number,
    a.appln_filing_date AS filing_date,
    a.appln_filing_year AS filing_year,
    a.granted,
    p.person_name AS applicant_name,
    p.person_ctry_code AS applicant_country
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_filing_year BETWEEN 2020 AND 2024
ORDER BY a.appln_filing_date DESC
LIMIT 100
""")

df_q1

---

## Query 2: Filter by Jurisdiction

Find patents from an applicant filed at specific patent offices.

**Try it:** Change the `AUTHORITIES` list to the jurisdictions you need (e.g. `('EP', 'US')` or `('CN', 'JP', 'KR')`).

In [ ]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
AUTHORITIES = "('EP', 'US', 'CN')"  # one or more: EP, US, CN, DE, JP, KR, WO ...
# --------------------

df_q2 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    COUNT(*) AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_auth IN {AUTHORITIES}
  AND a.appln_filing_year BETWEEN 2020 AND 2024
GROUP BY a.appln_auth
ORDER BY filings DESC
""")

df_q2

---

## Query 3: Detailed Filings per Jurisdiction and Year

See how many patents were filed per year at each selected office. Useful to spot trends.

In [ ]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
AUTHORITIES = "('EP', 'US', 'CN')" 
# --------------------

df_q3 = run_query(f"""
SELECT
    a.appln_auth AS authority,
    a.appln_filing_year AS filing_year,
    COUNT(*) AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_auth IN {AUTHORITIES}
  AND a.appln_filing_year BETWEEN 2015 AND 2024
GROUP BY a.appln_auth, a.appln_filing_year
ORDER BY a.appln_auth, a.appln_filing_year
""")

# Pivot into a readable table: years as rows, authorities as columns
df_q3_pivot = df_q3.pivot(index='filing_year', columns='authority', values='filings').fillna(0).astype(int)
df_q3_pivot

---

## Query 4: Patent Families for an Applicant

A patent family groups all filings worldwide that protect the **same invention**. This query shows how broadly an applicant protects its inventions.

Each row is one DOCDB family with:
- The earliest filing date (priority date)
- How many countries it was filed in (family size)
- Which authorities received filings

In [ ]:
# --- CHANGE THIS ---
APPLICANT = '%siemens%'
# -------------------

df_q4 = run_query(f"""
SELECT
    a.docdb_family_id,
    MIN(a.appln_filing_date) AS earliest_filing,
    a.docdb_family_size AS family_size,
    COUNT(DISTINCT a.appln_auth) AS nr_authorities,
    STRING_AGG(DISTINCT a.appln_auth, ', ' ORDER BY a.appln_auth) AS authorities
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND LOWER(p.person_name) LIKE '{APPLICANT}'
  AND a.appln_filing_year BETWEEN 2020 AND 2024
  AND a.docdb_family_id > 0
GROUP BY a.docdb_family_id, a.docdb_family_size
ORDER BY family_size DESC
LIMIT 100
""")

df_q4

---

## Query 5: Patent Families Filtered by Jurisdiction

Same as above, but only showing families that include filings in your selected jurisdictions.

**Example use case:** "Show me all Siemens inventions that were filed at both the EPO and in China."

In [ ]:
# --- CHANGE THESE ---
APPLICANT = '%siemens%'
MUST_INCLUDE_1 = 'EP'   # families must include this authority
MUST_INCLUDE_2 = 'CN'   # AND this authority
# --------------------

df_q5 = run_query(f"""
WITH family_auths AS (
    SELECT
        a.docdb_family_id,
        MIN(a.appln_filing_date) AS earliest_filing,
        MAX(a.docdb_family_size) AS family_size,
        COUNT(DISTINCT a.appln_auth) AS nr_authorities,
        STRING_AGG(DISTINCT a.appln_auth, ', ' ORDER BY a.appln_auth) AS authorities
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND LOWER(p.person_name) LIKE '{APPLICANT}'
      AND a.appln_filing_year BETWEEN 2020 AND 2024
      AND a.docdb_family_id > 0
    GROUP BY a.docdb_family_id
    HAVING
      COUNTIF(a.appln_auth = '{MUST_INCLUDE_1}') > 0
      AND COUNTIF(a.appln_auth = '{MUST_INCLUDE_2}') > 0
)
SELECT *
FROM family_auths
ORDER BY family_size DESC
LIMIT 100
""")

df_q5

---
# Part 2 &mdash; Profile an institution

Now a full profile of a single organisation, using a **university** as the example (TU&nbsp;Berlin). The pattern works for any applicant: first find the correct standardised name, then measure size, trend, families, filing strategy and technology profile.

---

## Query 1: Find Name Variants

**Why:** In PATSTAT, every organization exists under different spellings. Before you can analyze anything, you need to know how your university appears in the data.

**Tip:** Start with a broad search term (e.g. `'%berlin%'` and `'%technische%'`) and see what comes back.

**Try it:** Change `SEARCH_TERM` to match your university.

In [ ]:
# --- CHANGE THIS ---
SEARCH_TERM = '%technische%berlin%'
# -------------------

df_q1 = run_query(f"""
SELECT 
    p.person_name,
    p.psn_name,
    p.han_name,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND (   LOWER(p.person_name) LIKE '{SEARCH_TERM}'
       OR LOWER(p.han_name)    LIKE '{SEARCH_TERM}')
  AND a.docdb_family_id > 0
GROUP BY p.person_name, p.psn_name, p.han_name
HAVING COUNT(DISTINCT a.docdb_family_id) >= 3
ORDER BY families DESC
LIMIT 20
""")

df_q1

### What to Look For

You will likely see multiple rows for the same university with different spellings in `person_name`, but the same `psn_name`. For example:

- `Technische Universität Berlin` -> `TECHNISCHE UNIVERSITAET BERLIN`
- `TECHNISCHE UNIVERSITAET BERLIN` -> `TECHNISCHE UNIVERSITAET BERLIN`
- `TECHNISCHE UNIVERSITÄT BERLIN` -> `TECHNISCHE UNIVERSITAET BERLIN`

**Warning:** `han_name` may show incorrect values! For TU Berlin, it shows "TECHNISCHE UNIVERSITAT MUNCHEN" — this is a known PATSTAT error where the OECD harmonization merged TU Berlin and TU Munich. **Always use `psn_name` for universities.**

**Remember:** `psn_name = 'TECHNISCHE UNIVERSITAET BERLIN'` is the filter we'll use from now on.

---

## Query 2: Portfolio Overview – How Big Is It?

**Why:** Before diving into details, you want to know the basic numbers. How many inventions (families)? How many individual filings? Since when?

In [ ]:
# --- CHANGE THIS ---
PSN_NAME = 'TECHNISCHE UNIVERSITAET BERLIN'
# -------------------

df_q2 = run_query(f"""
SELECT 
    COUNT(DISTINCT a.docdb_family_id) AS patent_families,
    COUNT(DISTINCT a.appln_id)        AS individual_filings,
    MIN(a.appln_filing_year)          AS first_filing,
    MAX(a.appln_filing_year)          AS last_filing
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 1990 AND 2024
""")

df_q2

### How to Read This

For TU Berlin you would expect something like: ~507 inventions turned into ~1,268 filings worldwide. That means each invention was filed in ~2.5 countries on average.

---

## Query 3: Timeline – How Is Activity Developing?

**Why:** Spot trends. Is patent activity rising or declining? Are there notable years?

**Note:** The most recent 2 years (2023–2024) are likely incomplete due to publication delays.

In [ ]:
df_q3 = run_query(f"""
SELECT 
    a.appln_filing_year               AS year,
    COUNT(DISTINCT a.docdb_family_id) AS families,
    COUNT(DISTINCT a.appln_id)        AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY a.appln_filing_year
ORDER BY a.appln_filing_year
""")

df_q3

---

## Query 4: A Patent Family Under the Microscope

**Why:** Now it gets concrete! We take the largest family and look at all its members.

### What Is a Patent Family?

A DOCDB family groups all filings that protect the **same invention** — regardless of which country they were filed in.

- **EP** = European Patent
- **WO** = PCT (international) application
- **DE** = Germany, **US** = USA, **CN** = China, etc.

The query first finds the largest family, then displays all its members.

In [ ]:
df_q4 = run_query(f"""
WITH uni_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.docdb_family_id
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND p.psn_name = '{PSN_NAME}'
      AND a.docdb_family_id > 0
),
largest_family AS (
    SELECT docdb_family_id, COUNT(*) AS family_size
    FROM uni_apps
    GROUP BY docdb_family_id
    ORDER BY family_size DESC
    LIMIT 1
)
SELECT 
    lf.docdb_family_id,
    lf.family_size       AS total_members,
    a.appln_auth          AS authority,
    a.appln_nr            AS application_number,
    a.appln_filing_date   AS filing_date,
    a.granted,
    t.appln_title         AS title
FROM tls201_appln a
JOIN largest_family lf ON a.docdb_family_id = lf.docdb_family_id
LEFT JOIN tls202_appln_title t 
    ON a.appln_id = t.appln_id AND t.appln_title_lg = 'en'
WHERE a.docdb_family_id > 0
ORDER BY a.appln_filing_date, a.appln_auth
""")

df_q4

### How to Read This

An invention filed in 15+ countries signals high commercial significance. Look at:
- Which offices received filings (EP, WO, US, CN, JP...)
- Whether it was granted (Y/N) at each office
- The English title to understand what the invention is about

---

## Query 5: Filing Strategy – Which Countries?

**Why:** The filing strategy shows where the university wants to commercially exploit its inventions.

In [ ]:
df_q5 = run_query(f"""
SELECT 
    a.appln_auth AS authority,
    COUNT(DISTINCT a.docdb_family_id) AS families,
    COUNT(DISTINCT a.appln_id)        AS filings,
    SUM(CASE WHEN a.granted = 'Y' THEN 1 ELSE 0 END) AS granted
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY a.appln_auth
ORDER BY families DESC
LIMIT 15
""")

df_q5

### How to Interpret

For a typical German university:
- **EP + WO** dominate → The European and international (PCT) systems are the standard filing route
- **US** with high grant rate → Quality portfolio aimed at the US market
- **CN** selective → Only the most important inventions go to China
- **DE** (national) → Domestic filings, often the initial priority filing

---

## Query 6: Technology Profile – Which Fields?

**Why:** The technology profile reveals research strengths. We use the WIPO 35 technology fields for a clear overview.

In [ ]:
df_q6 = run_query(f"""
SELECT 
    tf.techn_sector    AS sector,
    tf.techn_field     AS technology_field,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
JOIN tls230_appln_techn_field atf ON a.appln_id = atf.appln_id
JOIN tls901_techn_field_ipc tf ON atf.techn_field_nr = tf.techn_field_nr
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY tf.techn_sector, tf.techn_field
ORDER BY families DESC
LIMIT 10
""")

df_q6

### How to Read This

TU Berlin is a broadly positioned technical university. You would expect:
- Strong in electrical engineering/IT (computer technology, digital communication, semiconductors)
- Significant in measurement and optics
- Surprisingly strong in medical technology and biotechnology

---
## Try it yourself

**Part 1** — change `APPLICANT` to any company (e.g. `'%bosch%'`, `'%airbus%'`).

**Part 2** — change `PSN_NAME` to your institution. Not sure of the exact name? Run Part 2 · Query 1 with a broad `SEARCH_TERM` (e.g. `'%karlsruhe%'`) first.

| University | `psn_name` |
|:-----------|:-----------|
| TU Munich | `TECHNISCHE UNIVERSITAET MUENCHEN` |
| KIT Karlsruhe | `KARLSRUHER INSTITUT FUER TECHNOLOGIE` |
| RWTH Aachen | `RHEINISCH WESTFAELISCHE TECHNISCHE HOCHSCHULE AACHEN` |
| TU Dresden | `TECHNISCHE UNIVERSITAET DRESDEN` |
| FU Berlin | `FREIE UNIVERSITAET BERLIN` |
| HU Berlin | `HUMBOLDT-UNIVERSITAET ZU BERLIN` |

### Handy tips
- Use `psn_name` for clean organisation matching (avoid `han_name`, which can merge different entities).
- Always add `applt_seq_nr > 0` to filter for applicants, not inventors.
- Count `docdb_family_id` to count inventions instead of individual filings.
- Add `AND a.appln_kind = 'A'` to exclude utility models and design patents.

### Where to go next
- **`3_querylib/`** — the Query Library: ready-to-use PATSTAT queries by question.
- **`4_patstat_explorer/`** — interactive applicant & technology search.
- **`5_lead_generation/`** — profile a whole region's applicants and segment them into lead tiers.
- **`2_legacy/`** — worked end-to-end examples (Airbus filing strategy, TU Dortmund portfolio).

---
*EPO Academy Training Material · mtc.berlin · depa.tech · PATSTAT Global, Autumn 2025*